# 1. Cargar el texto y convertir a índices ASCII

In [ ]:
import tensorflow as tf
import numpy as np
import os
import time

# 1. Descargar el dataset (Shakespeare)
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

# 2. Leer el texto y decodificar a ASCII
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Longitud del texto: {len(text)} caracteres')
print(f'Primeros 250 caracteres:\n{text[:250]}')

# 3. Crear el vocabulario (caracteres únicos)
vocab = sorted(set(text))
print(f'{len(vocab)} caracteres únicos en el vocabulario')

# 4. Mapeo de Caracteres a Números (y viceversa)
ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

def text_from_ids(ids):
    return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Longitud del texto: 1115394 caracteres
Primeros 250 caracteres:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

65 caracteres únicos en el vocabulario


# 2. Preparar el Dataset para la RNN

In [ ]:
# Convertir todo el texto a IDs
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

seq_length = 100  # Longitud de la secuencia de entrada
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

# Función para separar entrada (input) y objetivo (target)
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Batch y Shuffle para entrenamiento
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

print(dataset)

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>


# 3. Construir el Modelo RNN (usando GRU o LSTM)

In [ ]:
# Definición del modelo usando Keras Subclassing simplificado

class ShakespeareModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = self.embedding(inputs, training=training)

    if states is None:
        x, states = self.gru(x, training=training)
    else:
        x, states = self.gru(x, initial_state=states, training=training)

    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

model = ShakespeareModel(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=256,
    rnn_units=1024)

# 4. Entrenar el Modelo

In [ ]:
# 1. Modelo para ENTRENAMIENTO (Simple, sin devolver estados)
class TrainingModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    # stateful=False por defecto para training
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=False) # <--- OJO: False para entrenar simple
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, training=False):
    x = self.embedding(inputs, training=training)
    x = self.gru(x, training=training) # GRU maneja su estado interno sola
    x = self.dense(x, training=training)
    return x

model = TrainingModel(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=256,
    rnn_units=1024)

# 2. Compilar y Entrenar
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss)

# Prueba rápida de dimensiones
for input_batch, target_batch in dataset.take(1):
    pred = model(input_batch)
    print("Output shape:", pred.shape) # Debería ser (64, 100, vocab_size)

# Entrenar
history = model.fit(dataset, epochs=10)

Output shape: (64, 100, 66)
Epoch 1/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - loss: 3.0647
Epoch 2/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - loss: 1.9116
Epoch 3/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 50ms/step - loss: 1.6239
Epoch 4/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - loss: 1.4791
Epoch 5/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - loss: 1.3911
Epoch 6/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 1.3300
Epoch 7/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 1.2773
Epoch 8/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 1.2359
Epoch 9/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 1.1947
Epoch 10/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 1.1522


# 5. Generar Texto Nuevo

In [ ]:
# ==============================================================================
# 1. Modelo de GENERACIÓN (con reshape explícito del estado)
# ==============================================================================
class GenerativeModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)
    self.rnn_units = rnn_units

  def call(self, inputs, states, return_state=False, training=False):
    x = self.embedding(inputs, training=training)
    x, states = self.gru(x, initial_state=states, training=training)

    # FORZAR que el estado mantenga 2 dimensiones
    states = tf.reshape(states, [tf.shape(inputs)[0], self.rnn_units])

    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

# ==============================================================================
# 2. Instanciar y Copiar Pesos
# ==============================================================================
model_gen = GenerativeModel(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=256,
    rnn_units=1024)

dummy_input = tf.zeros((1, 1), dtype=tf.int64)
dummy_state = tf.zeros((1, 1024))
_ = model_gen(dummy_input, states=dummy_state)

model_gen.set_weights(model.get_weights())
print("✅ Pesos copiados exitosamente.")

# ==============================================================================
# 3. Clase OneStep (Simplificada y Robusta)
# ==============================================================================
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars
    self.rnn_units = 1024  # Guardamos las unidades

    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  def generate_one_step(self, inputs, states=None):
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Crear estado inicial con forma FIJA (1, 1024) si es None
    if states is None:
       states = tf.zeros([1, self.rnn_units], dtype=tf.float32)

    predicted_logits, states = self.model(inputs=input_ids, states=states, return_state=True)

    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    predicted_logits = predicted_logits + self.prediction_mask

    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)
    predicted_chars = self.chars_from_ids(predicted_ids)

    return predicted_chars, states

# ==============================================================================
# 4. Generar Texto
# ==============================================================================
one_step_model = OneStep(model_gen, chars_from_ids, ids_from_chars)

states = None
next_char = tf.constant(['ROMEO:'])
result = [next_char]

print("Generando texto...")
for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)


✅ Pesos copiados exitosamente.
Generando texto...
ROMEO:
To work? Youble dead from him;
But quiet itself may neble. You quarter'd blows,
To let with selloa-s will, and give themselves:
Therefore, one empurateous telliful,--
Our forbids he divide, the regal corse,
Either tempest man in the changict foe
He hath the duke and placking in the while.

PAULINA:
The senses of the true going slain:
Both though he was well, good day. O, hip lives!

MARCIUS:
Your highness with me they are.

GREMIO:
This is the seys.

CAMILLO:
But you may spoke to her
Your honour army.

CLAUDIO:
Flanders, she not too sleep:

All Hurdner:
Yet she she be wirdly run,
That he doth used to down, rost advance,
Advancely intercused with wear him?

BUCKINGHAM:
I hear love see, news ignoble dees: I have heard
The most orueness tooth, with what lips her-like
Hath suitors alone. Yet reason
At that envying leave, Rutland,
And brought-me from the town, only but he wakes,
To ann me into kisse. A save.

STANLEY:
Neight of you, f

# Metricas del Modelo

In [ ]:
# ==============================================================================
# MÉTRICAS DE CLASIFICACIÓN: Precision, Recall, F1-Score, Accuracy
# ==============================================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

print("="*80)
print("CALCULANDO MÉTRICAS DE CLASIFICACIÓN")
print("="*80)

# 1. Preparar datos de TEST
# Tomamos una porción del texto que el modelo no vio (últimos 100k caracteres)
test_size = 100000
test_text_raw = text[-test_size:]

print(f"\n📝 Preparando dataset de test con {test_size:,} caracteres...")

# Convertir el texto a IDs de caracteres
test_text_ids = ids_from_chars(tf.strings.unicode_split(test_text_raw, 'UTF-8'))

# Crear dataset de secuencias
test_dataset = tf.data.Dataset.from_tensor_slices(test_text_ids)

# Crear secuencias de longitud seq_length+1
seq_length = 100  # Ajusta según lo que usaste en entrenamiento
test_sequences = test_dataset.batch(seq_length+1, drop_remainder=True)

# Dividir en input y target
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

test_data = test_sequences.map(split_input_target)

# Crear batches
BATCH_SIZE = 64
test_batches = test_data.batch(BATCH_SIZE).prefetch(tf.data.experimental.AUTOTUNE)

# 2. Recopilar predicciones y etiquetas reales
all_predictions = []
all_labels = []

print("🔄 Evaluando modelo en datos de test...")
batch_count = 0

for input_batch, label_batch in test_batches:
    # Obtener predicciones del modelo
    predictions = model.predict(input_batch, verbose=0)

    # Convertir predicciones a clases (argmax)
    predicted_classes = np.argmax(predictions, axis=-1)

    # Aplanar las predicciones y etiquetas
    all_predictions.extend(predicted_classes.flatten())
    all_labels.extend(label_batch.numpy().flatten())

    batch_count += 1
    if batch_count % 10 == 0:
        print(f"   Procesados {batch_count} batches...")

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

print(f"\n✅ Total de predicciones evaluadas: {len(all_predictions):,}")

# 3. Calcular métricas globales
accuracy = accuracy_score(all_labels, all_predictions)

# Micro-average (equivalente a accuracy en clasificación multi-clase)
precision_micro = precision_score(all_labels, all_predictions, average='micro')
recall_micro = recall_score(all_labels, all_predictions, average='micro')
f1_micro = f1_score(all_labels, all_predictions, average='micro')

# Macro-average (promedio simple entre todas las clases)
precision_macro = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
recall_macro = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
f1_macro = f1_score(all_labels, all_predictions, average='macro', zero_division=0)

# Weighted-average (ponderado por frecuencia)
precision_weighted = precision_score(all_labels, all_predictions, average='weighted', zero_division=0)
recall_weighted = recall_score(all_labels, all_predictions, average='weighted', zero_division=0)
f1_weighted = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)

print("\n" + "="*80)
print("MÉTRICAS GLOBALES DEL MODELO")
print("="*80)

print(f"\n📊 ACCURACY (Precisión general): {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   → De cada 100 caracteres, el modelo predice correctamente {accuracy*100:.0f}")

print(f"\n📈 MÉTRICAS MICRO-AVERAGE:")
print(f"   • Precision: {precision_micro:.4f}")
print(f"   • Recall:    {recall_micro:.4f}")
print(f"   • F1-Score:  {f1_micro:.4f}")

print(f"\n📈 MÉTRICAS MACRO-AVERAGE:")
print(f"   • Precision: {precision_macro:.4f}")
print(f"   • Recall:    {recall_macro:.4f}")
print(f"   • F1-Score:  {f1_macro:.4f}")

print(f"\n📈 MÉTRICAS WEIGHTED-AVERAGE:")
print(f"   • Precision: {precision_weighted:.4f}")
print(f"   • Recall:    {recall_weighted:.4f}")
print(f"   • F1-Score:  {f1_weighted:.4f}")


CALCULANDO MÉTRICAS DE CLASIFICACIÓN

📝 Preparando dataset de test con 100,000 caracteres...
🔄 Evaluando modelo en datos de test...
   Procesados 10 batches...

✅ Total de predicciones evaluadas: 99,000

MÉTRICAS GLOBALES DEL MODELO

📊 ACCURACY (Precisión general): 0.6338 (63.38%)
   → De cada 100 caracteres, el modelo predice correctamente 63

📈 MÉTRICAS MICRO-AVERAGE:
   • Precision: 0.6338
   • Recall:    0.6338
   • F1-Score:  0.6338

📈 MÉTRICAS MACRO-AVERAGE:
   • Precision: 0.5800
   • Recall:    0.4576
   • F1-Score:  0.4878

📈 MÉTRICAS WEIGHTED-AVERAGE:
   • Precision: 0.6316
   • Recall:    0.6338
   • F1-Score:  0.6235
